# 01 – Synthetic Light-Curve Generation

This notebook demonstrates the synthetic signal generation module.
It reproduces the signal parameters from Table 3.1 of the thesis and
visualises the ground-truth light curve used in all experiments.

**Signal model:**
$$f_i = \mu_0 + A\sin\!\left(\frac{2\pi t_i}{P_0} + \phi_0\right) + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, \sigma_\varepsilon^2)$$


In [ ]:
import sys
sys.path.insert(0, '..')  # ensure src/ is importable from notebooks/

import numpy as np
import matplotlib.pyplot as plt
from src.simulation.generator import generate_synthetic_lightcurve
from src.utils.config import load_config

# Load configuration
cfg = load_config('../configs/experiment.yml')
sig_cfg = cfg['signal']
print('Signal parameters:', sig_cfg)

In [ ]:
# Generate the ground-truth signal
t, flux, params = generate_synthetic_lightcurve(**sig_cfg)

print(f'N cadences : {len(t)}')
print(f'Duration   : {t[-1]:.1f} days')
print(f'Flux range : [{flux.min():.4f}, {flux.max():.4f}]')
print(f'Flux mean  : {flux.mean():.4f}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Full time series
ax = axes[0]
ax.plot(t, flux, lw=0.6, color='#2166ac', alpha=0.85)
ax.set_xlabel('Time (days)')
ax.set_ylabel('Flux (normalised)')
ax.set_title(f'Synthetic light curve: N={len(t)}, P₀={sig_cfg["P0"]} d, A={sig_cfg["A"]}')
ax.grid(alpha=0.3)

# Phase-folded
ax = axes[1]
phase = (t % sig_cfg['P0']) / sig_cfg['P0']
sort_idx = np.argsort(phase)
ax.plot(phase[sort_idx], flux[sort_idx], '.', ms=1, color='#2166ac', alpha=0.5)
ax.set_xlabel('Phase')
ax.set_ylabel('Flux (normalised)')
ax.set_title(f'Phase-folded at P₀={sig_cfg["P0"]} d')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate different signal models
models = ['sinusoidal', 'multi_harmonic', 'eclipsing_binary', 'sawtooth']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, model in zip(axes.flat, models):
    _, f, _ = generate_synthetic_lightcurve(N=200, model=model, sigma_eps=0.005, seed=0)
    ax.plot(np.arange(200) * 0.0204, f, lw=1.0, color='#984ea3')
    ax.set_title(model)
    ax.set_xlabel('Time (days)')
    ax.set_ylabel('Flux')
    ax.grid(alpha=0.3)

plt.suptitle('Supported signal models', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Reproducibility check
_, f1, _ = generate_synthetic_lightcurve(N=100, seed=0)
_, f2, _ = generate_synthetic_lightcurve(N=100, seed=0)
_, f3, _ = generate_synthetic_lightcurve(N=100, seed=99)

print('Same seed identical:', np.allclose(f1, f2))
print('Different seed differs:', not np.allclose(f1, f3))